# Урок 8. Добавляем понятные столбцы в таблицу Titanic

Сегодня учимся **сохранять исходную таблицу и добавлять к ней полезные сведения, рассчитанные по уже известным данным**.

Наш вопрос: сколько указанных родственников путешествовало вместе с пассажиром? Можно ли разделить пассажиров на группы и сравнить наблюдения?

Не нужно знать новые команды заранее. Сначала вместе читаем маленький пример, запускаем его, объясняем результат и только затем повторяем приём на Titanic.

Новые английские названия столбцов будем каждый раз переводить. Они нужны, чтобы Python мог однозначно обратиться к нужным данным.

## Зачем мы преобразуем данные и создаём новые признаки

**Мы создаём новые признаки для последующего анализа данных и обучения модели машинного обучения.**

В исходной таблице сведения записаны так, как их собрали. Для нашего вопроса такое представление не всегда удобно. Мы хотим сначала исследовать, как характеристики пассажиров связаны с выживанием, а позже — проверить, можно ли по известным характеристикам прогнозировать исход.

Поэтому рассчитываем дополнительные характеристики: сколько родственников указано у пассажира, есть ли они вообще, к какой возрастной группе он относится. Такой рассчитанный столбец называют **новым признаком**. Мы преобразуем уже имеющиеся сведения в форму, удобную для конкретного вопроса.

На этом занятии мы только создаём понятные признаки. На последующих занятиях сравним группы, построим таблицы и графики; при изучении машинного обучения проверим, какие признаки помогают модели прогнозировать результат для новых наблюдений.

| Что создаём | Зачем это нужно для анализа | Что можно проверить позднее в модели |
|---|---|---|
| Family_size — пассажир и указанные родственники | Различается ли доля выживших при разном числе указанных родственников? | Помогает ли это число прогнозировать исход лучше? |
| Is_alone — без указанных родственников или с ними | Есть ли различие между двумя простыми группами? | Полезнее ли модели такой простой ответ «да/нет», чем подробное число? |
| Age_group — возрастная группа | Удобно ли сравнить младшую, старшую группу и записи с неизвестным возрастом в таблицах и на графиках? | Даёт ли группировка пользу по сравнению с исходным возрастом? |

**Полезность нового признака не гарантирована.** Возрастная группа упрощает представление, но теряет точность: 18 и 70 лет получают одну подпись. Модели может быть полезнее сам возраст. Поэтому сохраняем исходные столбцы и позднее проверяем пользу признаков на данных, которые не использовали для обучения.

Family_size и Is_alone описывают одни и те же исходные сведения по-разному; добавление обоих столбцов не создаёт новых исторических фактов и не обещает улучшения прогноза. Обнаруженная связь с выживанием также не доказывает причину.

**А зачем считаем число пассажиров?** Это контроль сохранности данных и основание для правильного расчёта долей. Например, «выжили 4 из 10» — 40%, а «4 из 100» — 4%. Общее число строк — проверка нашей работы, а не новый признак отдельного пассажира.

Цепочка занятия: **исследовательский вопрос → подходящие признаки → сравнение групп → осторожный вывод → позднее проверка пользы для прогноза**.


## Как работать с этим ноутбуком

Запускайте ячейки сверху вниз: выделите ячейку и нажмите Shift+Enter. Прочитайте текст перед кодом. В примерах уже есть готовые команды; в практике пишите собственный код в отведённой ячейке. После практики обязательно заполните «Мой ответ». Дополнительных ячеек без указания преподавателя не добавляйте.

Практика P1–P4 выполняется на занятии, H1–H3 — дома, в том же файле. Отсутствовавший выполняет обе части для ДЗ; это не создаёт зачёт присутствия или практики на занятии. Срок объявляет преподаватель отдельно.

Пустые учебные ячейки специально не вызывают ошибку: успешный запуск пустого шаблона не означает, что задания выполнены.

## 1. Вспоминаем, что хранит одна строка Titanic

Одна строка — один пассажир. Столбцы описывают разные сведения об этом человеке.

| Название в файле | По-русски | Что нужно помнить |
|---|---|---|
| PassengerId | Номер записи пассажира | По нему проверяем, что не потеряли и не продублировали человека |
| SibSp | Число братьев, сестёр, супругов на борту | Сам пассажир сюда **не входит** |
| Parch | Число родителей и детей на борту | Сам пассажир сюда **не входит** |
| Age | Возраст | Если значения нет, возраст нам неизвестен |
| Survived | Известный исход | 1 — выжил, 0 — не выжил |

Родственники в этих двух столбцах — не все возможные родственники и не все спутники. Например, друзья здесь не учитываются.

### Загрузка — готовая техническая часть

Следующая ячейка читает выданный CSV и проверяет, что используется нужная версия. Её код не требуется писать по памяти. После запуска появится таблица `df` — короткое имя нашей исходной таблицы в Python. `pd` — привычное сокращение имени библиотеки pandas, которая работает с таблицами. Если личного результата урока 7 нет, ячейка явно использует учебный исходник; прежнее ДЗ от этого не становится сданным.

В Colab загрузите titanic.csv через левую панель «Файлы». На локальном компьютере положите его рядом с notebook.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display
DATA_SHA = "4a437fde05fe5264e1701a7387ac6fb75393772ba38bb2c9c566405af5af4bd7"
paths=[Path("titanic.csv"),Path("data/titanic.csv")]
for base in [Path.cwd(), *Path.cwd().parents][:6]:
    paths += [base/"07_Данные"/"lesson-08"/"titanic.csv",base/"01_Course"/"Python для ИИ"/"07_Данные"/"lesson-08"/"titanic.csv"]
raw_path=next((p for p in paths if p.is_file()),None)
if raw_path is None: raise FileNotFoundError("Загрузите выданный titanic.csv рядом с notebook. В Colab: панель Файлы → загрузить.")
if hashlib.sha256(raw_path.read_bytes()).hexdigest()!=DATA_SHA: raise ValueError("Это другая версия исходника. Не заменяйте личный результат; отдельно загрузите выданный titanic.csv.")
raw=pd.read_csv(raw_path)
previous_name="titanic_lesson07.csv"
previous_paths=[Path("data")/previous_name,Path(previous_name),Path("../lesson-07/data")/previous_name]
previous_path=next((p for p in previous_paths if p.is_file()),None)
df=pd.read_csv(previous_path) if previous_path else raw.copy()
if not set(raw.columns).issubset(df.columns): raise ValueError("В личной таблице потеряны исходные поля. Восстановите предыдущий этап.")
if len(df)!=891 or not df.PassengerId.is_unique: raise ValueError("Изменён состав пассажиров: необходимо разобраться с предыдущим этапом.")
pd.testing.assert_frame_equal(df[raw.columns].sort_values("PassengerId").reset_index(drop=True),raw.sort_values("PassengerId").reset_index(drop=True),check_dtype=False)
if previous_path and list(df.columns)!=list(raw.columns):
    raise ValueError("Этап7 должен сохранять исходные столбцы без преобразований; проверьте личную версию.")
if previous_path: print("Используем личный предыдущий этап:",previous_path)
else: print("Личный предыдущий файл не найден. Учебный резерв восстановлен из исходника; это НЕ сдача предыдущего ДЗ. Восстановите свою работу отдельно.")
Path("data").mkdir(exist_ok=True)
stage_result=None
print("Строк:",len(df),"Выживших:",int(df.Survived.sum()),"Неизвестный возраст:",int(df.Age.isna().sum()))

Личный предыдущий файл не найден. Учебный резерв восстановлен из исходника; это НЕ сдача предыдущего ДЗ. Восстановите свою работу отдельно.
Строк: 891 Выживших: 342 Неизвестный возраст: 177


### Покажем только нужные нам поля

`df["SibSp"]` выбирает один столбец. Два уровня квадратных скобок, как ниже, выбирают **список из нескольких столбцов**. `.head()` показывает первые пять строк, а `display(...)` выводит их в виде таблицы. Это просмотр: значения `df` не меняются.

Посмотрите на любую строку и прочитайте её словами: «У этого пассажира в SibSp записано ..., в Parch записано ...».

In [2]:
# Выбираем пять интересующих нас столбцов и показываем первые пять пассажиров.
display(df[["PassengerId", "SibSp", "Parch", "Age", "Survived"]].head())

,PassengerId,SibSp,Parch,Age,Survived
0,1,1,0,22.0,0
1,2,1,0,38.0,1
2,3,0,0,26.0,1
3,4,1,0,35.0,1
4,5,0,0,35.0,0


## 2. Зачем нужна копия таблицы

**Для чего этот шаг в исследовании:** мы будем пробовать разные расчёты и правила. Копия позволяет проверить, что вывод не получен ценой случайного изменения исходных фактов. Сама копия не является новым признаком и не улучшает модель.

Представьте бумажную анкету. Если мы хотим сделать расчёты и пометки, удобно работать с её копией, сохранив оригинал для проверки. Так можно сравнить «до» и «после» и заметить случайное изменение.

В Python имя таблицы — не сама таблица. Запись `другое_имя = исходная_таблица` **не создаёт отдельную копию**: два имени обращаются к одному объекту. Поэтому используем `.copy()` — команду создания отдельной рабочей таблицы.

`work = df.copy()` читаем так: «возьми копию df и назови её work». Work переводится как «работа»: это имя рабочей таблицы. `.copy()` не прибавляет числа, не создаёт файл на компьютере и не добавляет новый столбец. Для наших обычных числовых и текстовых столбцов изменения рабочей таблицы не меняют исходную.

### Маленький пример: товар и цена

`pd.DataFrame(...)` создаёт таблицу из словаря: ключ — имя столбца, список — значения по строкам. Здесь мы сами задаём две строки, чтобы видеть каждое изменение.

In [3]:
# Исходная маленькая таблица: две строки с ценами.
goods = pd.DataFrame({"Товар": ["Тетрадь", "Ручка"], "Цена": [50, 30]})

# Создаём отдельную рабочую таблицу.
goods_work = goods.copy()

# Добавляем столбец только в рабочую таблицу.
# Берём цены и увеличиваем каждую на 10.
goods_work["Цена с доставкой"] = goods_work["Цена"] + 10

print("Исходная таблица:")
display(goods)
print("Рабочая копия:")
display(goods_work)

,Товар,Цена
0,Тетрадь,50
1,Ручка,30


,Товар,Цена,Цена с доставкой
0,Тетрадь,50,60
1,Ручка,30,40


Исходная таблица:
Рабочая копия:


**Что должно получиться:** в исходнике остались только «Товар» и «Цена». В копии появился третий столбец со значениями 60 и 40. Мы не меняли цену товара; добавили отдельный расчёт.

Разберём строку `goods_work["Цена с доставкой"] = goods_work["Цена"] + 10`:

1. Справа выбираем известные цены: 50 и 30.
2. Прибавляем к каждой 10: получаем 60 и 40.
3. Слева указываем, в какой столбец копии записать результат. Если такого столбца не было, pandas создаёт его.

**Проверьте понимание:** в какой таблице теперь три столбца? Что изменится, если вместо `+ 10` написать `+ 5`?

## Практика P1. Создаём рабочую копию Titanic

Зачем: дальше будем добавлять расчёты, а исходные сведения должны остаться для сравнения.

1. Создайте `work` командой `work = df.copy()`.
2. Покажите первые пять строк `work` через `display(work.head())`.
3. Выведите число строк в `df` и `work`: `len(df)` и `len(work)`.

`len(таблица)` означает количество строк, а не столбцов. Сразу после копирования таблицы должны иметь одинаковый вид и по 891 строке.

Копия пока ничем внешне не отличается от оригинала. Её смысл станет виден после добавления нового столбца.

In [4]:
# Выполните шаги из объяснения выше.
# Пишите по одной строке и проверяйте результат после каждого шага.
pass

### Объясните своими словами

Зачем мы сохраняем df без изменений? Создаст ли запись work = df отдельную копию? Почему?

**Мой ответ:** напишите здесь 2–3 предложения о своих результатах.

## 3. Считаем пассажира вместе с указанными родственниками: откуда берётся +1

**Для чего вводим Family_size:** хотим исследовать, различается ли выживаемость у пассажиров с разным числом указанных родственников. Новое число объединяет SibSp и Parch в один понятный показатель, включающий самого пассажира. Позднее можно проверить, помогает ли этот показатель модели прогнозировать исход; заранее этого не знаем.

Мы хотим получить **число людей: сам пассажир плюс родственники, указанные в двух полях**. Это поможет сравнивать поездку без указанных родственников и поездку с ними.

В `SibSp` и `Parch` посчитаны другие люди. Сам пассажир — человек, которому посвящена строка, — туда не включён. Поэтому к сумме добавляем **одного самого пассажира**.

| SibSp | Parch | Расчёт | Объяснение |
|---:|---:|---|---|
| 0 | 0 | 0 + 0 + 1 = 1 | Сам пассажир, указанных родственников нет |
| 1 | 2 | 1 + 2 + 1 = 4 | Пассажир и три указанных родственника |
| 1 | 0 | 1 + 0 + 1 = 2 | Пассажир и один указанный родственник |

**Важно:** `+1` относится к расчёту людей, а не к созданию копии таблицы. Это не новая метка и не техническое требование pandas. Если бы вопрос был «сколько указанных родственников без самого пассажира?», единицу прибавлять было бы не нужно.

Новый столбец назовём `Family_size` — условно «размер семейной группы по этим полям». Это не полная семья человека: таблица не сообщает обо всех родственниках, оставшихся дома или не учтённых этими полями.

In [5]:
# Три вымышленных пассажира: каждую строку легко посчитать вручную.
family_example = pd.DataFrame({"SibSp": [0, 1, 1], "Parch": [0, 2, 0]})

# Складываем соответствующие значения столбцов в каждой строке.
# Единица — сам пассажир, которого нет в SibSp и Parch.
family_example["Family_size"] = family_example["SibSp"] + family_example["Parch"] + 1
display(family_example)

,SibSp,Parch,Family_size
0,0,0,1
1,1,2,4
2,1,0,2


**Ожидаем:** в новом столбце 1, 4, 2. pandas выполняет правило сразу для всех строк. Отдельный цикл здесь не нужен.

**Повторите на Titanic в той же рабочей таблице:**

In [6]:
# Продолжение P1: добавьте Family_size по разобранной выше формуле.
# Затем покажите PassengerId, SibSp, Parch и Family_size.
# Сравните наличие Family_size в work.columns и df.columns.
pass

**Объясните словами:** почему у человека с двумя указанными родственниками получится 3? Почему число людей не может быть 0? В какой таблице появился новый столбец?

**Мой ответ:** напишите здесь свой ответ.

## 4. Отмечаем поездку без указанных родственников

**Для чего вводим Is_alone:** иногда нужен простой вопрос — «есть ли указанные родственники?», а не точное число. Деление на две группы облегчает сравнение долей и объяснение результата. Позднее проверим, полезно ли такое упрощение для модели. Оно не доказывает отсутствие друзей и не устанавливает причину выживания.

Мы уже умеем считать людей. Теперь хотим быстро находить строки, где в двух полях не указано ни одного родственника. Для этого добавим **столбец-ответ на вопрос**: «Число людей равно одному?»

Такое рассчитанное свойство строки часто называют **признаком**. Здесь признак — просто дополнительный столбец, а не специальный инструмент ИИ.

Назовём его `Is_alone` — «один?». В нём будут два ответа:

- `True` — «да»: Family_size равно 1, то есть указан только сам пассажир.
- `False` — «нет»: Family_size больше 1, то есть учтён хотя бы один родственник.

Это **не доказывает**, что человек вообще был один: он мог ехать с друзьями. Мы знаем только сведения из SibSp и Parch.

`=` записывает результат, а `==` сравнивает два значения. Справа от `=` сравниваем, слева называем столбец, куда сохраняем ответ.

In [7]:
# Для каждой строки спрашиваем: число людей равно 1?
family_example["Is_alone"] = family_example["Family_size"] == 1
display(family_example)

# value_counts() считает, сколько раз встретился каждый ответ.
display(family_example["Is_alone"].value_counts())

,SibSp,Parch,Family_size,Is_alone
0,0,0,1,True
1,1,2,4,False
2,1,0,2,False


Is_alone
False    2
True     1
Name: count, dtype: int64

**Ожидаем по строкам:** True, False, False. В подсчёте: False — 2, True — 1. Порядок вывода может отличаться: важны ответы и их количества.

### Как выбрать только нужных пассажиров

Выражение `family_example["Is_alone"] == True` даёт по одному ответу да/нет для каждой строки. Это список условий выбора — его ещё называют **булевой маской**. pandas оставит строки с True.

В записи `.loc[условие, столбцы]` слева от запятой выбираем **строки**, справа — **столбцы**. `:` справа означает «все столбцы».

In [8]:
# Сначала создаём условие выбора строк.
alone_example_rows = family_example["Is_alone"] == True
print("Для каких строк условие выполняется:")
print(alone_example_rows.tolist())

# Оставляем строки, для которых получили True; показываем все столбцы.
display(family_example.loc[alone_example_rows, :])

,SibSp,Parch,Family_size,Is_alone
0,0,0,1,True


Для каких строк условие выполняется:
[True, False, False]


### Зачем сравнивать группы и как считать долю

Можно задать исследовательский вопрос: «В нашей таблице среди какой группы доля выживших больше?» Для этого отдельно выбираем пассажиров каждой группы.

**Число выживших** и **доля выживших** отвечают на разные вопросы. Если из 10 человек выжили 4, доля — 4 / 10 = 0.4, или 40%. Если из 100 выжили 20, число больше, но доля меньше — 20%.

`Survived` хранит 0 или 1. Сумма этих чисел даёт число выживших. Делим эту сумму на число пассажиров выбранной группы, умножаем на 100 — получаем процент. Обязательно выводим и размер группы. Не утверждаем, что наличие родственников стало причиной результата: группы могут отличаться, например, по классу билета.

In [9]:
# На маленьком примере: пять наблюдений, три единицы.
outcomes_example = pd.Series([1, 0, 1, 0, 1])
example_total = len(outcomes_example)           # Всего наблюдений: 5.
example_survived = outcomes_example.sum()      # Выживших: 3.
example_percent = example_survived / example_total * 100
print("Всего:", example_total, "Выжили:", example_survived, "Доля, %:", example_percent)

Всего: 5 Выжили: 3 Доля, %: 60.0


## Практика P2. Добавляем отметку и сравниваем две группы

1. В work добавьте `Is_alone`: сравните `work["Family_size"]` с 1 через `==`.
2. Посчитайте ответы через `work["Is_alone"].value_counts()`.
3. Сохраните в `alone` строки, где Is_alone равно True, а в `with_relatives` — где оно равно False. Используйте `.loc[условие, :]` по примеру выше.
4. Для каждой группы выведите число строк, сумму Survived и процент: `сумма / число строк * 100`.

Обе группы вместе должны давать 891 пассажира. Названия alone и with_relatives — имена двух небольших таблиц, которые вы получите из work.

Пишите расчёты для групп отдельно. Не нужно одновременно изучать циклы, groupby и короткие способы записи.

In [10]:
# Выполните шаги из объяснения выше.
# Пишите по одной строке и проверяйте результат после каждого шага.
pass

### Объясните своими словами

Что означает True в Is_alone? Можно ли назвать его доказательством, что пассажир был без друзей? В какой группе у вас больше доля и какие числа это подтверждают?

**Мой ответ:** напишите здесь 2–3 предложения о своих результатах.

## 5. Разделяем пассажиров по известному возрасту

**Для чего вводим Age_group:** удобно сравнить возрастные группы и показать результаты в таблице или на графике. Это упрощение для анализа: мы теряем точные различия между возрастами внутри группы. Для будущей модели может оказаться полезнее исходный Age, поэтому его сохраняем. Неизвестный возраст выделяем отдельно, чтобы не выдавать отсутствие сведений за известный возраст.

Ещё один вопрос: «Сколько в таблице пассажиров младше 18 лет и сколько — от 18 лет?» Для ответа добавим `Age_group` — «группа по возрасту». Сам Age менять не будем: возраст 17.5 важнее и точнее, чем одна подпись «младше 18».

Нужны **три**, а не две группы:

1. «Младше18» — известный возраст строго меньше 18.
2. «От18» — известный возраст 18 или больше.
3. «Неизвестный» — возраста нет, поэтому мы не можем выбрать одну из первых двух групп.

Число 18 — выбранная нами граница для учебного сравнения. Мы не узнаём из таблицы юридический статус человека в 1912 году.

В примере ниже `float("nan")` создаёт обозначение отсутствующего числа. NaN — не ноль. Если возраст неизвестен, записать 0 означало бы выдумать, что это младенец.

### Добавляем подписи по шагам

Сначала всем строкам присваиваем подпись «Неизвестный». Затем меняем её только там, где есть известное значение, удовлетворяющее условию. Ровно 18 входит в «От18», потому что знак `>=` означает «больше или равно».

In [11]:
# Четыре понятных случая: меньше 18, ровно 18, старше 18 и неизвестный возраст.
age_example = pd.DataFrame({"Age": [17.9, 18, 40, float("nan")]})

# Создаём столбец с одинаковой начальной подписью для всех строк.
age_example["Age_group"] = "Неизвестный"
display(age_example)

# Условие < 18 выбирает только первую строку.
# Справа от запятой указано имя столбца, который хотим изменить.
age_example.loc[age_example["Age"] < 18, "Age_group"] = "Младше18"
display(age_example)

# Условие >= 18 выбирает строки со значениями 18 и 40.
age_example.loc[age_example["Age"] >= 18, "Age_group"] = "От18"
display(age_example)

,Age,Age_group
0,17.9,Неизвестный
1,18.0,Неизвестный
2,40.0,Неизвестный
3,NaN,Неизвестный


,Age,Age_group
0,17.9,Младше18
1,18.0,Неизвестный
2,40.0,Неизвестный
3,NaN,Неизвестный


,Age,Age_group
0,17.9,Младше18
1,18.0,От18
2,40.0,От18
3,NaN,Неизвестный


**Ожидаем в последней таблице:** Младше18, От18, От18, Неизвестный. Исходный Age остался прежним. Здесь Age прочитан как обычные числа с NaN: неизвестное значение не прошло ни одно из двух сравнений и сохранило начальную подпись.

`.isna()` проверяет отсутствие значения: возвращает True для неизвестного и False для известного. `.isna().sum()` считает количество неизвестных: при сложении True учитывается как 1, False как 0.

**Проверьте понимание:** куда попадёт 18? А 17.9? Почему неизвестный возраст нельзя автоматически назвать взрослым?

In [12]:
print("Какие возрасты неизвестны:")
print(age_example["Age"].isna().tolist())
print("Всего неизвестных:", age_example["Age"].isna().sum())
display(age_example["Age_group"].value_counts())

Age_group
От18           2
Младше18       1
Неизвестный    1
Name: count, dtype: int64

Какие возрасты неизвестны:
[False, False, False, True]
Всего неизвестных: 1


## Практика P3. Подписываем возрастные группы в Titanic

Повторите три строки из примера для work: сначала создайте Age_group с подписью «Неизвестный», затем измените подписи при Age < 18 и Age >= 18.

Покажите первые строки с Age и Age_group. Посчитайте группы через value_counts(). Отдельно выведите `work["Age"].isna().sum()`: должно остаться 177 неизвестных возрастов.

Чтобы проверить ровно 18 лет, выберите `work.loc[work["Age"] == 18, ["Age", "Age_group"]]`. Здесь справа список из двух столбцов. Посмотрите, какая подпись получилась.

Не заполняйте Age нулями и не заменяйте его подписями. Новый столбец хранит наше правило, исходный — известный возраст.

In [13]:
# Выполните шаги из объяснения выше.
# Пишите по одной строке и проверяйте результат после каждого шага.
pass

### Объясните своими словами

Какие три группы получились и сколько в каждой пассажиров? Почему Age=18 относится к «От18»? Как мы сохранили неизвестные значения?

**Мой ответ:** напишите здесь 2–3 предложения о своих результатах.

## 6. Проверяем, что исходные данные не испорчены

**Для чего проверяем число пассажиров:** чтобы при преобразованиях не потерять строки и правильно посчитать доли групп. Это проверка качества нашей работы; общее количество записей не является признаком одного пассажира.

Мы добавили три столбца. При этом не должны исчезнуть пассажиры, измениться их номера, известные возрасты или исходы. Это обычная проверка своей работы перед сохранением.

`work["Age"].equals(df["Age"])` сравнивает два столбца целиком и возвращает True, если совпадают значения, расположение строк и тип данных. В отличие от простого сравнения через `==`, этот способ корректно считает совпадающие пропуски одинаковыми. Мы не переставляли строки, поэтому здесь сравниваем их в исходном порядке.

`is_unique` — свойство столбца: True означает, что в нём нет повторяющихся значений. Для PassengerId это проверка номеров записей.

Ниже не нужно запоминать внутренние проверки pandas. Важно уметь объяснить, **что именно** вы проверили.

## Практика P4. Проверяем таблицу перед сохранением

Выведите по очереди:

1. `len(work)` — сколько пассажиров осталось: ожидаем 891.
2. `work["PassengerId"].is_unique` — нет ли повторяющихся номеров: ожидаем True.
3. `work["PassengerId"].equals(df["PassengerId"])` — те ли пассажиры и тот ли порядок: True.
4. `work["Survived"].equals(df["Survived"])` — не изменились ли исходы: True.
5. `work["Age"].equals(df["Age"])` — не изменились ли возрасты и пропуски: True.
6. `"Family_size" in df.columns` — не добавился ли расчёт в исходник: False.

Если результат отличается, вернитесь к соответствующему шагу. После проверки выполните `stage_result = work.copy()`. Это просто имя для готовой к сохранению копии; оно нужно готовой служебной ячейке в конце.

Наличие трёх новых столбцов не доказывает правильность формул. Проверьте несколько строк вручную, особенно нули в SibSp/Parch и возраст ровно 18.

In [14]:
# Выполните шаги из объяснения выше.
# Пишите по одной строке и проверяйте результат после каждого шага.
pass

### Объясните своими словами

Какие проверки дали True, а какая должна дать False? Доказывают ли эти проверки, что наличие родственников помогает выжить?

**Мой ответ:** напишите здесь 2–3 предложения о своих результатах.

## Домашняя работа: закрепляем те же действия

Здесь не требуется изучать новые сложные команды. Пользуйтесь разобранными примерами и собственными ячейками практики. Если на занятии что-то не успели, сначала завершите этот шаг. Срок преподаватель объявляет отдельно.

### H1. Что изменится, если разделить возрастные группы по 16 годам вместо 18?

Зачем: понять, что подпись зависит от выбранного нами правила. Сам возраст человека при этом не меняется. Например, 17-летний окажется в разных группах при двух правилах, а 10-летний — в младшей при обоих.

1. Создайте отдельную копию `h = work.copy()`, чтобы не менять готовые столбцы work.
2. Добавьте в h `Age_group16`: сначала «Неизвестный», затем «Младше16» при Age < 16 и «От16» при Age >= 16. Повторите образец Age_group, заменив границу и имя столбца.
3. Посчитайте группы через value_counts().
4. Найдите строки с возрастом от 16 включительно до 18 не включительно: `(h["Age"] >= 16) & (h["Age"] < 18)`. Знак `&` означает «оба условия одновременно». Скобки вокруг каждого сравнения обязательны.
5. Выведите число выбранных строк через len. Именно у этих известных возрастов поменяется принадлежность к младшей или старшей группе. Сами разные тексты «От16» и «От18» сравнивать для подсчёта переходов нельзя.
6. По образцу P2 сравните две группы **известных** возрастов: младше 16 и от 16. Для каждой назовите число пассажиров и долю выживших. Неизвестные остаются отдельно.

Проверка вручную: 15 лет — младшая при обоих правилах; 17 — меняет группу; 18 — старшая при обоих; неизвестный — остаётся неизвестным.

In [15]:
# H1. Повторите знакомый пример с новой границей 16.
pass

**Объясните своими словами:** какой возраст меняет группу, почему это происходит и означает ли смена подписи изменение исходных фактов? Назовите свои числа и проценты.

**Мой ответ:** напишите здесь свой ответ.

### H2. Проверяем фразу «Is_alone=False означает, что пассажир ехал всей семьёй»

Зачем: научиться не обещать больше, чем известно из данных. False означает только, что число людей по нашей формуле больше одного. Таблица не сообщает, сколько родственников у человека вообще и кто остался дома.

Выберите пять строк, где Is_alone равно False. По образцу `.loc` покажите PassengerId, SibSp, Parch, Family_size; затем `.head()` оставит первые пять строк. Для одной строки объясните формулу вручную. Напишите более точную фразу вместо «всей семьёй».

In [16]:
# H2. Найдите пять пассажиров с указанными родственниками.
pass

**Мой ответ:** объясните одну строку и исправьте утверждение.

### H3. Поясняем добавленные столбцы и сохраняем результат

Зачем: другой человек должен понять вашу таблицу, не читая весь код. Ваша работа должна открыться повторно, а не только выглядеть готовой на экране.

Заполните эту таблицу своими словами:

| Новый столбец | На какой вопрос отвечает | Как посчитан | Что из него нельзя узнать |
|---|---|---|---|
| Family_size | ... | ... | ... |
| Is_alone | ... | ... | ... |
| Age_group | ... | ... | ... |

Напишите короткое пояснение в `data/README_lesson08.md`: источник, число пассажиров, три правила и что вы сделали с неизвестным возрастом. README — обычный текстовый файл с пояснением. Его можно создать в редакторе; расширение .md позволяет оформлять текст заголовками и таблицами.

Затем выполните готовую ячейку сохранения ниже и убедитесь, что она сообщила об успешной повторной загрузке. Ваша дополнительная группа Age_group16 из H1 остаётся в notebook; основной файл для следующего урока содержит исходные поля и Family_size, Is_alone, Age_group.

In [17]:
# H3. Заполните поясняющую таблицу в Markdown выше.
# Создайте свой README_lesson08.md по описанному образцу.
pass

## Сохранение: что делает готовая ячейка

`stage_result` — готовая таблица из P4. Следующий код сохраняет её в CSV без лишнего столбца с номерами строк, затем читает CSV обратно и сравнивает данные. Это проверяет, что результат можно открыть повторно. Код также создаёт вспомогательные файлы с количеством известных значений и сведениями об источнике; писать эти технические команды по памяти не нужно.

Если P4 ещё не сделано, сохранения не будет: сначала завершите практику. Успешное сохранение не проверяет ваши письменные ответы и само по себе не означает зачёт.

В Colab папка `data` находится во временной среде. Скачайте её содержимое и свой notebook до завершения сессии.

In [18]:
# Служебная передача результата. Наличие файлов не подтверждает выполнение всех заданий.
if not isinstance(stage_result,pd.DataFrame):
    print("Экспорт не выполнен: сначала завершите P1–P4 и назначьте stage_result.")
else:
    pd.testing.assert_frame_equal(stage_result[raw.columns].sort_values("PassengerId").reset_index(drop=True),raw.sort_values("PassengerId").reset_index(drop=True),check_dtype=False)
    target=Path("data")/"titanic_lesson08.csv"
    stage_result.to_csv(target,index=False,encoding="utf-8")
    back=pd.read_csv(target)
    pd.testing.assert_frame_equal(stage_result,back,check_dtype=False)
    quality=pd.DataFrame({"field":stage_result.columns,"known":stage_result.count().to_numpy(),"missing":stage_result.isna().sum().to_numpy()})
    quality.to_csv(Path("data")/"quality_lesson08.csv",index=False)
    manifest={"lesson":8,"source_sha256":DATA_SHA,"previous":str(previous_path) if previous_path else "учебный резерв, личный этап отсутствовал","output":target.name,"sha256":hashlib.sha256(target.read_bytes()).hexdigest(),"rows":len(back),"columns":list(back.columns)}
    (Path("data")/"manifest_lesson08.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Сохранено и повторно проверено:",target)
    print("Скачайте data из Colab и загрузите в homework/lesson-08/data. Добавьте собственный README. Сам notebook сдайте как solution.ipynb.")

Экспорт не выполнен: сначала завершите P1–P4 и назначьте stage_result.


## Перед сдачей

1. Сохраните notebook с выполненным кодом, результатами и своими ответами.
2. На GitHub положите его в `homework/lesson-08/solution.ipynb`.
3. В `homework/lesson-08/data/` положите созданные файлы и свой README_lesson08.md.
4. На следующий урок понадобится `titanic_lesson08.csv`.
5. После перезапуска запустите notebook сверху вниз: он должен воспроизвести ваши вычисления без старых переменных.

Умею ли я объяснить: зачем копия; кого учитывает +1; что означает True; почему неизвестный возраст не ноль; что проверил перед сохранением?

### Откуда взята идея занятия

Практика работы с таблицами и осознанного обращения с пропусками опирается на Reuven M. Lerner, *Pandas Workout* (2024), главы 2 и 5. Маленькие примеры и объяснения написаны для этого курса. На этом занятии не нужны новые библиотеки, автоматическое заполнение возраста и сложные оптимизации. Исследуется учебная таблица пассажиров Titanic, а не полный список всех людей на судне.